# Phase II, Pair 1: question E, out-of-distribution contexts (MOCKUP)

*A draft of the structure, to be filled in and merged into `05_question_e.ipynb`. Everything in italics is a proposal, change it.*

| | |
|---|---|
| Authors | Sandro Müller K52010874 |
| Status | mockup |
| Phase I result this builds on | *(check the exact experiment in the experiments README: the injected-anomaly experiments and the band-honesty result)* |

## 1 · Question

*Does TiRex-2 notice, in its internal states or in its forecast band, when a context is far from typical weather, and how large must an anomaly be before it does?*

## 2 · Why this question

*In Phase I the band was honest in normal weather (81 % inside) but not at fronts (32 %). We do not know whether the model "knows" when a context is unusual, or whether it fails silently: the error grows but the band does not. "Out of distribution" is too vague to test, so we turn it into concrete anomalies of growing size.*

## 3 · Hypothesis and what would count as an answer

**Hypothesis.**

- **H1.** The distance of the hidden state from typical states grows with the size of the anomaly.
- **H2.** The forecast band gets wider with the size of the anomaly.
- **H3.** For some anomaly types the model fails silently: the error grows, but the state and the band do not react.

*What would count as an answer:*
- *Used / noticed: distance and band width rise clearly above the level of unchanged contexts.*
- *Silent failure: error rises, distance and band stay at the level of unchanged contexts.*
- *Control (must not move): the whole context scaled and shifted (a unit change). The model standardises each series, so this should be invisible. If it moves, something is wrong.*

Verdicts: [H1](#h1) · [H2](#h2) · [H3](#h3)

## 4 · Setup

*Imports, paths, model loading. The state function (`hidden_states`) is the one from question F; decide with Carlos whether to move it into a shared helper before copying it.*

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from tirex2 import TimeseriesType, load_model

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
DATA = Path(os.environ.get("AIVIS_DATA", ROOT / "data" / "austria_15km"))
RESULTS = ROOT / "data" / "notebooks" / "results" / "05_question_e"
RESULTS.mkdir(parents=True, exist_ok=True)

CONTEXT, HORIZON = 672, 24  # Phase I design: 28 days of history, 24 h forecast
SEED = 0

model = load_model(os.environ.get("TIREX_CKPT", "NX-AI/TiRex-2"), device=os.environ.get("TIREX_DEVICE", "cpu"))
core = model.model  # the TiRex2 backbone

## 5 · Data

*Cases: start times and points as in Phase I (the file `tirex2_forecasts.npz` has the start times). A normal case is the unchanged context; every anomaly is added to a copy of a normal context. Say how many cases, and why that is enough.*

In [ ]:
# TODO: load grid_long.parquet and tirex2_forecasts.npz, choose N starts x M points (as in question F),
# build the filled context SER[hour, point] and the observed truth OBS[hour, point].
# TODO: choose the "typical" reference contexts (for example the normal starts of Phase I)

### Anomalies

*Each function takes a context `x` (672 hourly values, °C) and returns an anomalous copy. The size is the parameter we vary.*

| Anomaly | What it imitates | Parameter | Expected |
|---|---|---|---|
| `spike` | a single wrong value | size in °C | small effect |
| `stuck` | a sensor stuck at one value | hours | should be noticed for long stretches |
| `gap` | missing data | hours | the model masks NaN, may stay silent |
| `level_shift` | a sensor offset in the last hours | hours, size in °C | strong effect |
| `unit_change` | the whole series rescaled (control) | none | **must not move** |

In [ ]:
def spike(x, size, position=-1):
    """One wrong value: add `size` to one hour (default: the last hour of the context)."""
    y = x.copy()
    y[position] += size
    return y


def stuck(x, hours):
    """A stuck sensor: the last `hours` hours all hold the value from just before."""
    y = x.copy()
    y[-hours:] = x[-hours - 1]
    return y


def gap(x, hours):
    """Missing data: the last `hours` hours are NaN (the model masks NaN)."""
    y = x.copy()
    y[-hours:] = np.nan
    return y


def level_shift(x, hours, size):
    """A sensor offset: add `size` to the last `hours` hours."""
    y = x.copy()
    y[-hours:] += size
    return y


def unit_change(x):
    """Control: the whole series rescaled (here: °C to °F). The model standardises, so nothing should change."""
    return x * 1.8 + 32


ANOMALIES = {
    "spike": [(s,) for s in (2, 5, 10, 20)],  # size in °C
    "stuck": [(h,) for h in (6, 12, 24, 72)],  # hours
    "gap": [(h,) for h in (6, 12, 24, 72)],  # hours
    "level_shift": [(24, s) for s in (2, 5, 10, 20)],  # last 24 h, size in °C
    "unit_change": [()],  # control
}
FUNCS = {"spike": spike, "stuck": stuck, "gap": gap, "level_shift": level_shift, "unit_change": unit_change}

## 6 · Technique 1: distance of the state from typical states

*What it shows: how far the model's internal state moves away from the states of normal contexts. What it cannot show: whether the model uses that information in the forecast.*

*Plan: for each anomaly and size, take the last-block state of the last patch, and compute the distance to its nearest typical states (k nearest neighbours). One number per case.*

In [ ]:
# TODO: hidden_states(ts) as in question F (one forward pass, output of every block)
# TODO: typical_states = last-block state of the last patch for the normal reference contexts
# TODO: def distance_score(state, k=10): mean distance to the k nearest typical states

## 7 · Technique 2: band width and error

*What it shows: whether the model's own uncertainty (the 0.1 to 0.9 band) grows with the anomaly, and how the error behaves. What it cannot show: why.*

*Plan: for each anomaly and size, forecast, then compute the mean band width (q0.9 minus q0.1) and the MAE against the observed truth.*

In [ ]:
def band_width(pred):
    """Mean width of the 0.1 to 0.9 band over the forecast hours. pred: [9 quantiles, HORIZON]."""
    return float(np.mean(pred[8] - pred[0]))


# TODO: forecast the anomalous contexts with model.forecast, then band_width(...) and MAE against OBS

## 8 · Results per hypothesis

*One part per hypothesis, each with the numbers it needs and a verdict, as in question F.*

<a id="h1"></a>
### 8.1 · H1: the state distance grows with the anomaly

*Evidence: distance score against anomaly size, per type. Control (`unit_change`) must stay at the level of unchanged contexts.*

**Verdict:** *to fill*

<a id="h2"></a>
### 8.2 · H2: the band gets wider with the anomaly

**Verdict:** *to fill*

<a id="h3"></a>
### 8.3 · H3: silent failure for some types

*Evidence: types where the error rises but the distance and the band do not.*

**Verdict:** *to fill*

In [ ]:
# TODO: tables per anomaly type: size | distance | band width | MAE, next to the unchanged contexts as the reference

## 9 · Checks

*Does the control stay put? Do the results hold on other dates and points? Is the distance score itself reliable (for example: do unchanged contexts get a low score on a held-out part)?*

In [ ]:
# TODO: control check, split by normal vs front starts, and a held-out check of the distance score

## 10 · Figure decisions

| Figure | Keep / change / drop | Used for | Why |
|---|---|---|---|
| Dose-response lines: distance against size, one line per type | *planned* | H1 | |
| Band width and error against size | *planned* | H2 | |
| Scatter: distance against band width, coloured by type | *planned* | H3 | silent failures sit at high error, low distance |

## 11 · What it means

*Answer to the question, in plain words: does the model notice, and how large must the anomaly be? What is still open?*

## 12 · What did not work

*Dead ends and negative results. They are part of the record.*